---
## Setup — Environment, MCP Servers & Memory Stores

This cell initialises everything needed for the notebook:
- Connects to **3 MCP servers** (CRM, Memory, Postgres)
- Creates memory stores (ST, LT) and the agent
- All CRM operations go through the **nawaloka-crm MCP server**
- Memory tools are available via **nawaloka-memory MCP server**

In [1]:
import sys, os, re, time, asyncio, nest_asyncio
sys.path.insert(0, "../src")

# Allow async MCP client inside Jupyter
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv()

from infrastructure.log import setup_logging
from loguru import logger
setup_logging("INFO", for_notebook=True)

# ── Memory stores (direct imports — used for teaching memory internals) ──
from infrastructure.llm import get_chat_llm, get_extractor_llm, get_default_embeddings
from infrastructure.config import ST_MAX_TURNS, ST_TTL_SECONDS
from memory.st_store import ShortTermMemoryStore
from memory.lt_store import LongTermMemoryStore
from memory.memory_ops import MemoryRecaller, MemoryDistiller
from memory.schemas import ConversationTurn
from memory.episodic_store import EpisodicMemoryStore
from memory.episodic_store import create_episode_from_turns
from memory.procedural_store import ProceduralMemoryStore

embedder = get_default_embeddings()
llm = get_chat_llm(temperature=0)
llm_extractor = get_extractor_llm(temperature=0)

st_store = ShortTermMemoryStore()
lt_store = LongTermMemoryStore(embedder)
recaller = MemoryRecaller(st_store, lt_store)
distiller = MemoryDistiller(llm_extractor, lt_store)

# ── Build the MCP-backed agent ─────────────────────────────────
from agents.orchestrator import build_agent_mcp

agent = asyncio.get_event_loop().run_until_complete(
    build_agent_mcp(enable_rag=True, enable_web=True)
)

# ── Helpers ────────────────────────────────────────────────────
def extract_phone(text: str) -> str:
    match = re.search(r"\+?[\d][\d\s\-\.\(\)]{7,}[\d]", text)
    return re.sub(r"[\s\-\.\(\)]", "", match.group()) if match else "anonymous"

def show_response(resp, label=""):
    if label: print(f"\n{'='*60}\n{label}\n{'='*60}")
    print(f"Route   : {resp.route}" + (f" / {resp.action}" if resp.action else ""))
    print(f"Latency : {resp.latency_ms}ms")
    print(f"Answer  : {resp.answer[:500]}")

USER_ID = "94781030736"
SESSION_ID = "nb01-demo"

logger.success("Environment ready (MCP-backed agent)")
print(f"MCP tools discovered ({len(agent.mcp_tools)} total):")
for name in agent.mcp_tools:
    print(f"  - {name}")

2026-04-11 23:36:41.327 | DEBUG    | infrastructure.observability:<module>:166 - langfuse package not installed — @observe is a no-op.


ℹ️ ✓ Supabase SQL engine created
ℹ️ ✅ Supabase connection test: SUCCESS
ℹ️ ✅ pgvector extension: INSTALLED
ℹ️ ✓ Schema validation passed: vector(1536)
ℹ️ Connecting to MCP servers: ['nawaloka-crm', 'nawaloka-memory']
ℹ️ Loaded 11 tools via MCP: ['lookup_patient', 'search_doctors', 'create_booking', 'cancel_booking', 'reschedule_booking', 'recall_context', 'get_recent_turns', 'add_turn', 'search_facts', 'store_fact', 'list_facts']
ℹ️ CRM tool backed by nawaloka-crm MCP server
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
ℹ️ Connected to Qdrant Cloud at https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"
ℹ️ ✓ CAG cache ready (Qdrant collection='cag_cache', dim=1536, threshold=0.90)
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.

### 3e · `web_search` Route — Real-Time Information
For questions needing live data (current hours, events, news), the router dispatches to **Tavily Web Search**.

In [2]:
resp = agent.chat(
    user_message="What are the current visiting hours at Nawaloka Hospital?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show_response(resp, "WEB SEARCH route — Tavily real-time")

ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ Retrieved 5 facts from LT memory for user 94781030736
ℹ️ Recalled 4 ST turns, 3 LT facts for user 94781030736
ℹ️ LangFuse client initialised (host=https://us.cloud.langfuse.com)
ℹ️ Distilling new facts for 94781030736...
ℹ️ Upserted 3 facts to LT memory (0 new, 3 merged)
ℹ️ Distilled 3 facts for user 94781030736

WEB SEARCH route — Tavily real-time
Route   : web_search
Latency : 29275ms
Answer  : Anushka, the general visiting hours at Nawaloka Hospital are typically from **10:00 AM to 8:00 PM**.

However, please note that specific visiting hours might vary depending on the ward or department, and there could be updates. For the most accurate and up-to-date information, I recommend checking the official Nawaloka Hospital website or calling them directly at **+94 11 544 4444**.


---
## Section 4 — Memory System: 4 Layers

The agent has 4 memory types working together:

| Layer | What It Stores | Retrieval | Backend |
|---|---|---|---|
| **Short-Term** | Recent conversation turns | Last N turns (recency) | Supabase `st_turns` |
| **Long-Term Semantic** | Distilled facts & preferences | Cosine similarity (pgvector) | Supabase `mem_facts` |
| **Episodic** | Full conversation sessions | Cosine similarity on summary | Supabase `mem_episodes` |
| **Procedural** | Step-by-step workflows | Cosine similarity | Supabase `mem_procedures` |

### 4a · Short-Term Memory — Conversation Ring Buffer

- **Capacity**: Ring buffer — last `ST_MAX_TURNS=30` turns per user/session
- **TTL**: 24 hours (configurable)
- **Purpose**: Conversational continuity within and across sessions

In [3]:
MEM_USER    = USER_ID
MEM_SESSION = "nb01-memory-demo"

# Build a realistic conversation about Anushka's medical profile
conversation = [
    ("user",      "Hi, I'm Anushka. I have a cardiac stress test coming up."),
    ("assistant", "Hello Anushka! I can see that in your records. How can I help you prepare?"),
    ("user",      "From now on, remember that I take atenolol 50mg every morning for blood pressure."),
    ("assistant", "Got it! I'll remember your atenolol 50mg daily medication schedule."),
    ("user",      "I'm allergic to penicillin — please always remember this."),
    ("assistant", "Noted! Penicillin allergy recorded. This is critical safety information."),
    ("user",      "Also remind me that I have a meniscus tear follow-up with orthopedics."),
    ("assistant", "Remembered! Orthopedics follow-up for the meniscus tear is noted."),
]

print(f"Storing {len(conversation)} turns in short-term memory...\n")
for role, content in conversation:
    turn = ConversationTurn(
        user_id=MEM_USER, session_id=MEM_SESSION,
        role=role, content=content, ts=time.time(),
    )
    st_store.append(turn, ST_MAX_TURNS, ST_TTL_SECONDS)
    emoji = "👤" if role == "user" else "🤖"
    print(f"  {emoji} [{role:9s}]: {content}")
    time.sleep(0.05)

logger.success(f"\n{len(conversation)} turns stored in Supabase (st_turns)")

Storing 8 turns in short-term memory...

  👤 [user     ]: Hi, I'm Anushka. I have a cardiac stress test coming up.
  🤖 [assistant]: Hello Anushka! I can see that in your records. How can I help you prepare?
  👤 [user     ]: From now on, remember that I take atenolol 50mg every morning for blood pressure.
  🤖 [assistant]: Got it! I'll remember your atenolol 50mg daily medication schedule.
  👤 [user     ]: I'm allergic to penicillin — please always remember this.
  🤖 [assistant]: Noted! Penicillin allergy recorded. This is critical safety information.
  👤 [user     ]: Also remind me that I have a meniscus tear follow-up with orthopedics.
  🤖 [assistant]: Remembered! Orthopedics follow-up for the meniscus tear is noted.
✅ 
8 turns stored in Supabase (st_turns)


In [4]:
recent = st_store.recent(MEM_USER, MEM_SESSION, k=10)

print(f"Retrieved {len(recent)} turns from ST memory:\n")
for i, t in enumerate(recent, 1):
    emoji = "👤" if t.role == "user" else "🤖"
    age = f"{time.time() - t.ts:.0f}s ago"
    print(f"  {i}. {emoji} [{t.role:9s}] ({age}): {t.content}")

Retrieved 10 turns from ST memory:

  1. 👤 [user     ] (915s ago): Also remind me that I have a meniscus tear follow-up with orthopedics.
  2. 🤖 [assistant] (913s ago): Remembered! Orthopedics follow-up for the meniscus tear is noted.
  3. 👤 [user     ] (22s ago): Hi, I'm Anushka. I have a cardiac stress test coming up.
  4. 🤖 [assistant] (20s ago): Hello Anushka! I can see that in your records. How can I help you prepare?
  5. 👤 [user     ] (17s ago): From now on, remember that I take atenolol 50mg every morning for blood pressure.
  6. 🤖 [assistant] (12s ago): Got it! I'll remember your atenolol 50mg daily medication schedule.
  7. 👤 [user     ] (10s ago): I'm allergic to penicillin — please always remember this.
  8. 🤖 [assistant] (7s ago): Noted! Penicillin allergy recorded. This is critical safety information.
  9. 👤 [user     ] (5s ago): Also remind me that I have a meniscus tear follow-up with orthopedics.
  10. 🤖 [assistant] (3s ago): Remembered! Orthopedics follow-up for the m

---
## Section 5 — Memory Distillation: LLM Extracts Long-Term Facts

The **MemoryDistiller** watches the conversation and extracts important facts when:
- Turn count ≥ 5, OR
- Conversation contains keywords: `remember`, `from now on`, `remind me`, `always`, `never`

**Process:**
```
Conversation Turns
    → LLM (llama-3.1-8b-instant / Groq)  ← ultra-fast extraction
    → JSON array of {text, tags}
    → score_memory_fact()  (0.5×recency + 0.3×repetition + 0.2×explicitness)
    → dedupe_facts()  (NumPy cosine matrix within batch, threshold=0.85)
    → LongTermMemoryStore.upsert()  (pgvector dedup, threshold=0.92)
```

In [5]:
should = distiller.should_distill(recent)

print(f"Should distill? {should}")
print(f"\nTrigger analysis:")
print(f"  Turn count    : {len(recent)} (threshold: 5)")
for kw in ["remember", "from now on", "remind me", "always", "never"]:
    hit = any(kw in t.content.lower() for t in recent)
    print(f"  '{kw}'  : {hit}")

Should distill? True

Trigger analysis:
  Turn count    : 10 (threshold: 5)
  'remember'  : True
  'from now on'  : True
  'remind me'  : True
  'always'  : True
  'never'  : False


In [6]:
print("Running memory distillation (LLM call)...\n")
facts = distiller.distill(MEM_USER, recent)

logger.success(f"Extracted {len(facts)} long-term facts:")
for i, f in enumerate(facts, 1):
    print(f"\n  {i}. {f.text}")
    print(f"     Score : {f.score:.3f}")
    print(f"     Tags  : {', '.join(f.tags)}")

Running memory distillation (LLM call)...

ℹ️ Upserted 4 facts to LT memory (0 new, 4 merged)
ℹ️ Distilled 4 facts for user 94781030736
✅ Extracted 4 long-term facts:

  1. Anushka has a cardiac stress test coming up
     Score : 0.587
     Tags  : appointment, cardiac_stress_test, orthopedics

  2. Anushka takes atenolol 50mg daily for blood pressure
     Score : 0.587
     Tags  : medication, atenolol, blood_pressure, schedule

  3. Anushka is allergic to penicillin
     Score : 0.587
     Tags  : allergy, penicillin, allergic_reaction

  4. Anushka has a meniscus tear follow-up with orthopedics
     Score : 0.587
     Tags  : appointment, meniscus_tear, orthopedics, follow_up


In [7]:
# Verify facts are semantically searchable via pgvector
test_queries = ["blood pressure medication", "drug allergies", "orthopedics follow-up"]

print("Semantic search over long-term memory (pgvector cosine):")
for q in test_queries:
    results = lt_store.query(MEM_USER, q, k=3, threshold=0.3)
    print(f"\n  Query: '{q}'  →  {len(results)} fact(s)")
    for r in results:
        print(f"    ↳ {r.text}  [score={r.score:.3f}]")

Semantic search over long-term memory (pgvector cosine):
ℹ️ Retrieved 3 facts from LT memory for user 94781030736

  Query: 'blood pressure medication'  →  3 fact(s)
    ↳ Patient takes Atenolol 50mg daily medication  [score=0.587]
    ↳ Anushka takes Atenolol 50mg every morning for blood pressure  [score=0.587]
    ↳ Remind Anushka to take Atenolol 50mg daily for blood pressure  [score=0.627]
ℹ️ Retrieved 3 facts from LT memory for user 94781030736

  Query: 'drug allergies'  →  3 fact(s)
    ↳ User is allergic to penicillin  [score=0.587]
    ↳ Anushka needs to inform her doctor about her allergies  [score=0.587]
    ↳ Anushka needs to inform medical staff about penicillin allergy at appointments  [score=0.587]
ℹ️ Retrieved 3 facts from LT memory for user 94781030736

  Query: 'orthopedics follow-up'  →  3 fact(s)
    ↳ Contact Nawaloka Hospital for cardiac rehabilitation protocol  [score=0.587]
    ↳ Patient name is required to check upcoming appointments  [score=0.587]
    ↳ Remind

---
## Section 6 — Long-Term Semantic Recall: Token-Budgeted Retrieval

The **MemoryRecaller** combines ST and LT memories within a **token budget**:

```
max_tokens = 500
  ├─ Short-term budget : 60% = 300 tokens  (recent turns, max 4)
  └─ Long-term budget  : 40% = 200 tokens  (semantic facts, max 3)
```

LT facts are retrieved via **cosine similarity** against the user's query — so the most *relevant* facts surface, not just the most recent.

In [8]:
query = "What medications does Anushka take and does she have any allergies?"

st_turns, lt_facts = recaller.recall(
    user_id=MEM_USER, session_id=MEM_SESSION,
    query=query, k_st=6, k_lt=5, max_tokens=500,
)

print(f"Recalled: {len(st_turns)} ST turns  +  {len(lt_facts)} LT facts")

st_tokens = sum(recaller.count_tokens(t.content) for t in st_turns)
lt_tokens = sum(recaller.count_tokens(f.text)    for f in lt_facts)
used      = st_tokens + lt_tokens
denom     = used if used > 0 else 1  # guard against zero-division

print(f"\nToken budget:")
print(f"  Total   : {used}/500")
print(f"  ST (60%): {st_tokens} tokens ({st_tokens/denom*100:.1f}% of used)")
print(f"  LT (40%): {lt_tokens} tokens ({lt_tokens/denom*100:.1f}% of used)")

# format_context only accepts st_turns; build LT section manually
st_context = recaller.format_context(st_turns)
if lt_facts:
    lt_lines = ["=== REMEMBERED FACTS ==="]
    for i, f in enumerate(lt_facts, 1):
        lt_lines.append(f"{i}. {f.text}  [{', '.join(f.tags)}]")
    lt_context = "\n".join(lt_lines)
else:
    lt_context = ""

full_context = "\n".join(part for part in [st_context, lt_context] if part)

print(f"\nMemory context passed to LLM:")
print("-" * 60)
print(full_context)

ℹ️ Retrieved 5 facts from LT memory for user 94781030736
ℹ️ Recalled 4 ST turns, 3 LT facts for user 94781030736
Recalled: 4 ST turns  +  3 LT facts

Token budget:
  Total   : 93/500
  ST (60%): 63 tokens (67.7% of used)
  LT (40%): 30 tokens (32.3% of used)

Memory context passed to LLM:
------------------------------------------------------------
=== RECENT CONVERSATION ===
User: I'm allergic to penicillin — please always remember this.
Assistant: Noted! Penicillin allergy recorded. This is critical safety information.
User: Also remind me that I have a meniscus tear follow-up with orthopedics.
Assistant: Remembered! Orthopedics follow-up for the meniscus tear is noted.

=== REMEMBERED FACTS ===
1. Anushka's medication and allergy information is unknown  [medication, allergy, unknown]
2. Anushka has allergies  [allergy, unknown]
3. Anushka needs to consult doctor or medical staff for medication and allergy info  [medication, allergy, consultation, medical_history]


---
## Section 7 — Episodic Memory: Full Conversation Snapshots

Episodic memory stores **complete sessions** — not just extracted facts.

- **Semantic vs Episodic**: `"User takes atenolol 50mg"` (semantic fact) vs the full 8-turn conversation where this was discussed, with an LLM-generated summary (episode).
- **Use case**: *"What did we discuss last Tuesday?"* → episodic search returns the session.

In [9]:
episodic_store = EpisodicMemoryStore(embedder)

# Create episode from the conversation turns (LLM generates summary)
episode = create_episode_from_turns(
    user_id=MEM_USER, session_id=MEM_SESSION,
    turns=recent, llm=llm,
)

print(f"Episode created:")
print(f"  Session : {episode.session_id}")
print(f"  Turns   : {episode.turn_count}")
print(f"  Topics  : {', '.join(episode.topic_tags)}")
print(f"  Summary : {episode.summary}")

episodic_store.store_episode(episode)
logger.success("Episode stored in Supabase pgvector")

# Query episodic memory
for q in ["medication discussion", "allergy information"]:
    eps = episodic_store.query_episodes(MEM_USER, q, k=2, threshold=0.3)
    print(f"\n  Query '{q}' → {len(eps)} episode(s)")
    for ep in eps:
        print(f"    Topics: {', '.join(ep.topic_tags)} | {ep.turn_count} turns")
        print(f"    Summary: {ep.summary[:120]}...")

ℹ️ ✓ Episodic memory store initialized (Supabase/pgvector)
Episode created:
  Session : nb01-memory-demo
  Turns   : 10
  Topics  : medication, allergy
  Summary : The user is providing the assistant with various medical information, including an upcoming meniscus tear follow-up, a cardiac stress test, a daily medication (atenolol 50mg), and a penicillin allergy, all of which the assistant acknowledges and records.
ℹ️ ✓ Stored episode baa9e8fc-ea44-4fca-9bcc-47ca3e29d1a5 with 10 turns
✅ Episode stored in Supabase pgvector
ℹ️ Retrieved 2 episodes for user 94781030736

  Query 'medication discussion' → 2 episode(s)
    Topics: medication, allergy | 10 turns
    Summary: The user is providing the assistant with various medical information, including an upcoming meniscus tear follow-up, a c...
    Topics: medication, allergy | 10 turns
    Summary: The user is providing the assistant with various medical information, including an upcoming meniscus tear follow-up, a c...
ℹ️ Retrieved 2 epis

---
## Section 8 — Procedural Memory: Workflow Guidance

Procedural memory stores **step-by-step workflows** pre-seeded by an admin. When a user query matches a procedure, the agent retrieves and follows the steps — ensuring consistency and explainability.

Example: `"Book an appointment"` → retrieves `book_new_appointment` procedure → follows 8 steps.

In [10]:
from memory.prompts import format_procedures

proc_store = ProceduralMemoryStore()
query      = "I want to book an appointment with a cardiologist"

procedures = proc_store.query_procedures(query, top_k=2, threshold=0.3)

if procedures:
    for p in procedures:
        logger.success(f"Matched: {p.name}  (similarity={p.similarity:.2f})")
        print(f"  Category    : {p.category}")
        print(f"  Description : {p.description}")
        print(f"  Steps       : {len(p.steps)}")
    print("\nFormatted for agent prompt:")
    print("=" * 60)
    print(format_procedures(procedures))
else:
    logger.warning("No procedures found. Run: python scripts/seed_procedures.py")

✅ Matched: book_new_appointment  (similarity=0.53)
  Category    : booking
  Description : Book a new appointment for a patient with a doctor at a location
  Steps       : 8
✅ Matched: find_doctor  (similarity=0.50)
  Category    : inquiry
  Description : Help patient find a suitable doctor by specialty or name
  Steps       : 5

Formatted for agent prompt:

**Procedure 1: book_new_appointment** (booking)
Description: Book a new appointment for a patient with a doctor at a location
When to use: When a patient requests to schedule a new appointment

Steps:
  1. identify_patient: Verify patient identity using phone number or patient ID. If not found, suggest registration.
  2. identify_requirements: Ask for: specialty needed, preferred doctor (if any), preferred location, date/time preferences, reason for visit.
  3. check_availability: Query CRM database for available doctors matching specialty and location within requested timeframe.
  4. present_options: Present 2-3 available time slo

---
## Section 9 — Without vs. With Memory: The Impact of Context

This is the core demonstration of why memory matters. The **same LLM** gives:
- A **generic, useless** answer without memory
- A **personalised, accurate** answer with recalled context

Same model. Same query. Different context → fundamentally different response.

In [11]:
compare_query = "What medications does Anushka take and does she have any allergies?"

# ── Without memory ─────────────────────────────────────────────
print("=" * 72)
print("WITHOUT memory (raw LLM — no context):")
print("=" * 72)
baseline = llm.invoke(compare_query)
baseline_text = baseline.content if hasattr(baseline, "content") else str(baseline)
print(baseline_text)

# ── Rebuild full_context if this cell is run independently ─────
# (st_turns / lt_facts come from the recall cell above)
st_ctx = recaller.format_context(st_turns)
if lt_facts:
    lt_lines = ["=== REMEMBERED FACTS ==="]
    for i, f in enumerate(lt_facts, 1):
        lt_lines.append(f"{i}. {f.text}  [{', '.join(f.tags)}]")
    lt_ctx = "\n".join(lt_lines)
else:
    lt_ctx = ""
full_context = "\n".join(part for part in [st_ctx, lt_ctx] if part)
injected_tokens = used  # set in the recall cell above

prompt = f"{full_context}\n\nUSER QUERY: {compare_query}\n\nAnswer based on the above:"

print("\n" + "=" * 72)
print(f"WITH memory ({injected_tokens} tokens of context injected):")
print("=" * 72)
memory_resp = llm.invoke(prompt)
memory_text = memory_resp.content if hasattr(memory_resp, "content") else str(memory_resp)
print(memory_text)

print("\n" + "=" * 72)
logger.error("WITHOUT memory: LLM refuses — no context about this user")
logger.success(f"WITH memory ({injected_tokens} tokens): LLM gives precise, personalised answer")

WITHOUT memory (raw LLM — no context):
I cannot provide you with information about Anushka's medications or allergies. That kind of information is private medical data and I do not have access to it.

WITH memory (93 tokens of context injected):
Anushka's medication and allergy information is unknown. She needs to consult a doctor or medical staff for this information.

❌ WITHOUT memory: LLM refuses — no context about this user
✅ WITH memory (93 tokens): LLM gives precise, personalised answer


---
## Section 10 — Multi-Turn: Progressive Memory Building

Watch memory **accumulate** across turns. Each turn adds to short-term memory. The distiller extracts long-term facts. By the end, the agent knows the user deeply — and the memory persists **across sessions** (LT facts survive in pgvector).

In [12]:
MULTI_SESSION = "nb01-multiturn"
first_msg = "Hi, I'm Anushka. My mobile is 078 103 0736. I have a cardiac stress test coming up."
MULTI_USER = extract_phone(first_msg)

messages = [
    first_msg,
    "I take atenolol 50mg every morning for blood pressure. Please remember this.",
    "I'm allergic to penicillin — very important, always remember!",
    "Can you find me a cardiologist?",
    "What is the medication administration policy at the hospital?",
    "What do you remember about my health profile and medications?",
]

print("Multi-Turn Conversation — Progressive Memory Building")
print("=" * 72)

for i, msg in enumerate(messages, 1):
    print(f"\n{'─' * 72}")
    print(f"Turn {i}: {msg}")
    print("─" * 72)

    resp = agent.chat(user_message=msg, user_id=MULTI_USER, session_id=MULTI_SESSION)

    ctx_lines = len(resp.memory_context.strip().split("\n")) if resp.memory_context.strip() else 0
    print(f"  Route          : {resp.route}" + (f" / {resp.action}" if resp.action else ""))
    print(f"  Memory context : {ctx_lines} lines (grows each turn)")
    print(f"  Latency        : {resp.latency_ms}ms")
    print(f"  Answer         : {resp.answer[:300]}{'...' if len(resp.answer) > 300 else ''}")

print(f"\n{'=' * 72}")
logger.success("Multi-turn complete. Memory context grew from 0 → N lines across turns.")
logger.info("Long-term facts now persisted in pgvector — will survive across sessions.")

Multi-Turn Conversation — Progressive Memory Building

────────────────────────────────────────────────────────────────────────
Turn 1: Hi, I'm Anushka. My mobile is 078 103 0736. I have a cardiac stress test coming up.
────────────────────────────────────────────────────────────────────────
ℹ️ Retrieved 5 facts from LT memory for user 0781030736
ℹ️ Recalled 4 ST turns, 3 LT facts for user 0781030736
ℹ️ Distilling new facts for 0781030736...
ℹ️ Upserted 2 facts to LT memory (0 new, 2 merged)
ℹ️ Distilled 2 facts for user 0781030736
  Route          : direct
  Memory context : 9 lines (grows each turn)
  Latency        : 15954ms
  Answer         : Hello Anushka! It's good to hear from you. I've noted your name and mobile number.

You mentioned you have a cardiac stress test coming up. Is there anything specific you'd like to know or any way I can assist you regarding your upcoming test?

────────────────────────────────────────────────────────────────────────
Turn 2: I take atenolol 50m

---
## Section 11 — Observability with LangFuse

Every `.chat()` call creates a **LangFuse trace** with nested spans:

```
agent_chat  (trace)
  ├─ memory_recall        (span)       ← ST + LT retrieval time
  ├─ router               (generation) ← gpt-4o-mini tokens + cost
  ├─ tool_dispatch        (span)
  │    └─ crm | rag | web_search  (span)
  ├─ synthesiser          (generation) ← gemini-2.5-flash tokens + cost
  ├─ memory_store         (span)
  └─ memory_distill       (span)
       └─ distill_facts   (generation) ← llama-3.1-8b tokens + cost
```

Open your **LangFuse dashboard** → Traces to see per-step latency, token usage, cost, and I/O for every call.

In [13]:
from infrastructure.observability import flush

flush()  # Push pending events to LangFuse

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
print(f"LangFuse dashboard: {host}")
print("  Traces → filter by tag 'agent' → see full pipeline waterfall.")
print("  Each trace: recall → route → dispatch → synthesise → store → distill")

LangFuse dashboard: https://cloud.langfuse.com
  Traces → filter by tag 'agent' → see full pipeline waterfall.
  Each trace: recall → route → dispatch → synthesise → store → distill


---
## Summary

| Component | Role | Key File |
|---|---|---|
| **QueryRouter** | LLM classifies intent → `crm \| rag \| web_search \| direct` | `agents/router.py` |
| **CRMTool** | Patient lookup, doctor search, booking CRUD | `agents/tools/crm_tool.py` |
| **RAGTool** | CAG cache → CRAG → Qdrant KB | `agents/tools/rag_tool.py` |
| **WebSearchTool** | Tavily real-time search | `agents/tools/web_search_tool.py` |
| **ShortTermStore** | Ring buffer (30 turns, 24h TTL) | `memory/st_store.py` |
| **LongTermStore** | Semantic facts (pgvector, cosine KNN) | `memory/lt_store.py` |
| **EpisodicStore** | Full session snapshots (pgvector) | `memory/episodic_store.py` |
| **ProceduralStore** | Workflow steps (pgvector) | `memory/procedural_store.py` |
| **MemoryDistiller** | LLM extracts facts from turns | `memory/memory_ops.py` |
| **MemoryRecaller** | Token-budgeted hybrid recall | `memory/memory_ops.py` |
| **LangFuse** | Traces every step (latency, tokens, cost) | `infrastructure/observability.py` |

---

**Next →** Notebook 02 rebuilds this pipeline as a **LangGraph StateGraph** with MCP-backed tools, the Supervisor-Worker pattern, typed shared state, and persona-specialized sub-agents.